# Feature Extraction - Flickr8k

Notebook ini mengekstrak feature vector dari setiap gambar di dataset Flickr8k pakai InceptionV3 yang sudah pretrained di ImageNet (tanpa classification head). Hasilnya disimpan ke disk supaya tidak perlu dijalankan lagi setiap kali training.

In [24]:
import os, json
import numpy as np
from pathlib import Path


def _find_root(marker="requirements.txt"):
    p = Path(os.getcwd())
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError("Repo root tidak ketemu, pastikan requirements.txt ada di root.")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
print("Working dir:", REPO_ROOT)

Working dir: c:\Users\Farrel's Laptop\Desktop\mlml\CNN-RNN-digaspolndangakndangak


In [25]:
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))

import tensorflow as tf
from tensorflow import keras
from shared.image_utils import extract_and_cache_features

## Config

Sesuaikan path di bawah kalau lokasi dataset berbeda.

In [26]:
FLICKR_DIR   = "data/flickr8k/Images"
FEATURES_DIR = "features"
FEATURES_NPY = os.path.join(FEATURES_DIR, "flickr8k_features.npy")
IDX_JSON     = os.path.join(FEATURES_DIR, "flickr8k_idx.json")

TARGET_SIZE  = (299, 299)   # input size InceptionV3
BATCH_SIZE   = 32

os.makedirs(FEATURES_DIR, exist_ok=True)

## Load CNN encoder

InceptionV3 dengan `pooling='avg'` menghasilkan vector 2048 dimensi per gambar. Semua weight-nya di-freeze karena kita hanya butuh feature extraction, bukan fine-tuning.

In [27]:
base = keras.applications.InceptionV3(
    include_top=False,
    pooling="avg",
    weights="imagenet",
)
base.trainable = False
print("Output shape:", base.output_shape)

Output shape: (None, 2048)


## Kumpulkan semua path gambar

Path diurutkan supaya index mapping-nya konsisten setiap kali dijalankan.

In [28]:
all_images = sorted(os.listdir(FLICKR_DIR))
all_paths  = [os.path.join(FLICKR_DIR, f) for f in all_images]
image_ids  = [os.path.splitext(f)[0] for f in all_images]

print(f"Total gambar: {len(all_paths)}")
print("Contoh path:", all_paths[:3])

Total gambar: 8091
Contoh path: ['data/flickr8k/Images\\1000268201_693b08cb0e.jpg', 'data/flickr8k/Images\\1001773457_577c3a7d70.jpg', 'data/flickr8k/Images\\1002674143_1b742ab4b8.jpg']


## Ekstrak dan cache feature

`extract_and_cache_features` cek dulu apakah file `.npy` sudah ada sebelum jalan forward pass. Kalau sudah ada langsung di-load. Pertama kali jalan butuh beberapa menit.

In [29]:
features = extract_and_cache_features(
    image_paths=all_paths,
    keras_encoder=base,
    cache_path=FEATURES_NPY,
    batch_size=BATCH_SIZE,
    target_size=TARGET_SIZE,
)
print("Shape features:", features.shape)  # (N, 2048)

Shape features: (8091, 2048)


## Simpan index mapping

Kita simpan dict `{image_id: row_index}` supaya notebook lain bisa cari feature vector berdasarkan nama file gambar (tanpa ekstensi).

In [30]:
idx_map = {img_id: i for i, img_id in enumerate(image_ids)}

with open(IDX_JSON, "w") as f:
    json.dump(idx_map, f)

print(f"Index map disimpan ke {IDX_JSON}")
print("Contoh entry:", list(idx_map.items())[:3])

Index map disimpan ke features\flickr8k_idx.json
Contoh entry: [('1000268201_693b08cb0e', 0), ('1001773457_577c3a7d70', 1), ('1002674143_1b742ab4b8', 2)]


## Sanity check

In [31]:
sample_id  = image_ids[0]
sample_idx = idx_map[sample_id]
vec        = features[sample_idx]

print("Image ID  :", sample_id)
print("Feature   :", vec[:5], "...")
print("Min / Max :", vec.min().round(4), "/", vec.max().round(4))

Image ID  : 1000268201_693b08cb0e
Feature   : [0.11328042 0.15111338 0.7328461  0.31158835 0.8352995 ] ...
Min / Max : 0.0 / 1.7808
